In [25]:
# Using the company facts API, we can get a ton of data about many companies
# But tags aren't always the same across companies, and can vary a lot
# Let's analyze this

In [2]:
import pandas as pd
import os
import scraper
import json
import matplotlib.pyplot as plt

In [4]:
data_dir = f"data/{os.listdir("data")[-1]}" # most recent scrape

In [5]:
comp_dict = scraper.get_companies_dict()

# First look

In [ ]:
# First, let's check what types of taxonomies / fact categories there are
# We will put them in a dictionary to count their occurences

taxonomies_dict = {}

for cik in os.listdir(data_dir):
    comp = comp_dict[cik]
    if not os.path.exists(f"{data_dir}/{cik}/companyfacts.json"): continue

    with open(f"{data_dir}/{cik}/companyfacts.json", "r") as file:
        content = file.read()
        if content == "" or content == "\n": continue

        facts = json.loads(content)
        for taxon in facts["facts"].keys():
            if taxon not in taxonomies_dict: taxonomies_dict[taxon] = 1
            else: taxonomies_dict[taxon] += 1

Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x0000024AFF691BE0>>
Traceback (most recent call last):
  File "C:\Users\marco\AppData\Roaming\Python\Python313\site-packages\ipykernel\ipkernel.py", line 796, in _clean_thread_parent_frames
    active_threads = {thread.ident for thread in threading.enumerate()}
  File "c:\Users\marco\AppData\Local\Programs\Python\Python313\Lib\threading.py", line 1479, in enumerate
    def enumerate():
KeyboardInterrupt: 


In [ ]:
taxonomies_dict

{'dei': 454, 'us-gaap': 415, 'invest': 130, 'srt': 162, 'ifrs-full': 90}

In [ ]:
# dei taxonomy: "Document and Entity Information". Usually only conaints 1-2 things like number of shares.
# us-gaap taxonomy: the standard, probably contains the most data
# invest taxonomy: Seems to barely have anything.
# srt taxonomy: SEC Reporting Taxonomy. Seems to barely have anything.
# ifrs-full taxonomy: Another kind of taxonomy. Seems big, but not very popular.

In [ ]:
# Now let's look at tag frequency.
# I will prepend dei: to dei tags and us-gaap: to us-gaap tags

tags_dict = {}
num_comps = 0

for cik in os.listdir(data_dir):
    comp = comp_dict[cik]
    if not os.path.exists(f"{data_dir}/{cik}/companyfacts.json"): continue

    with open(f"{data_dir}/{cik}/companyfacts.json", "r") as file:
        content = file.read()
        if content == "" or content == "\n": continue
        facts = json.loads(content)

        num_comps += 1
        for taxon in facts["facts"]:
            for tag in facts["facts"][taxon]:
                tag_name = taxon + ":" + tag
                if tag_name not in tags_dict: tags_dict[tag_name] = 1
                else: tags_dict[tag_name] += 1

In [ ]:
# Now show the results, but sorted by occurences decreasing, save in a txt file
print(f"Number of companies checked is {num_comps}")
print(f"Number of unique tags is {len(tags_dict)}")
items = sorted(tags_dict.items(), key = lambda tup: tup[1], reverse=True)

with open("tags.txt", "w") as file:
    file.write("\n".join([str(i) for i in items]))

Number of companies checked is 470
Number of unique tags is 10194


In [ ]:
# Observations:
    # The main issue is how spread out and messed up the tags are

    # e.g. some companies, instead of using us-gaap:<tag>, may use a combination of 
    # us-gaap:<subtag> which add up to <tag>

    # Similarly, some companies don't have the data in us-gaap but do have it in ifrs-full

    # So I need to find ways to clean up the data and make the data more consistent

# Tag analyzing

## CommonStockSharesOustanding

In [ ]:
# Relevent tags:
    # ifrs-full:NumberOfSharesOutstanding
    # dei:EntityCommonStockSharesOustanding
    # us-gaap:CommonStockSharesOutstanding
    # us-gaap:WeightedAverageNumberOfSharesOutstandingBasic
    # ifrs-full:WeightedAverageShares

# The dei tag is very common, but some stocks don't have it! (43 stocks)
# In that case: replace with us-gaap:CommonStockSharesOutstanding (27 stocks remaining)
# As a second backup, use ifrs-full:NumberOfSharesOutstanding (22 stocks remaining)
# Next, use us-gaap:WeightedAverageNumberOfSharesOutstandingBasic (7 stocks remaining)
# Next, use ifrs-full:WeightedAverageShares (1 stock remaining)
# For this final company (BP PLC) I couldn't find any tags referring to the number of shares

for cik in os.listdir(data_dir):
    comp = comp_dict[cik]
    if not os.path.exists(f"{data_dir}/{cik}/companyfacts.json"): continue

    with open(f"{data_dir}/{cik}/companyfacts.json", "r") as file:
        content = file.read()
        if content == "" or content == "\n": continue
        facts = json.loads(content)["facts"]

        if (("dei" not in facts or "EntityCommonStockSharesOutstanding" not in facts["dei"])
            and ("us-gaap" not in facts or "CommonStockSharesOutstanding" not in facts["us-gaap"])
            and ("ifrs-full" not in facts or "NumberOfSharesOutstanding" not in facts["ifrs-full"])
            and ("us-gaap" not in facts or "WeightedAverageNumberOfSharesOutstandingBasic" not in facts["us-gaap"])
            and ("ifrs-full" not in facts or "WeightedAverageShares" not in facts["ifrs-full"])
            
        ):
            print(f"tags not found in {comp.name}")

tags not found in BP PLC


In [ ]:
# Seeing as I'm using weighted average shares to increase tag consistency,
# what if i just use weighted average shares?

In [18]:
for cik in os.listdir(data_dir):
    comp = comp_dict[cik]
    if not os.path.exists(f"{data_dir}/{cik}/companyfacts.json"): continue

    with open(f"{data_dir}/{cik}/companyfacts.json", "r") as file:
        content = file.read()
        if content == "" or content == "\n": continue
        facts = json.loads(content)["facts"]

        if (
            ("us-gaap" not in facts or "WeightedAverageNumberOfSharesOutstandingBasic" not in facts["us-gaap"])
            and ("ifrs-full" not in facts or "WeightedAverageShares" not in facts["ifrs-full"])
            and ("ifrs-full" not in facts or "AdjustedWeightedAverageShares" not in facts["ifrs-full"])
        ):
            print(f"tags not found in {comp.name}")

tags not found in BP PLC
tags not found in TotalEnergies SE
tags not found in MEXICAN ECONOMIC DEVELOPMENT INC
tags not found in PETROBRAS - PETROLEO BRASILEIRO SA
tags not found in Energy Transfer LP
tags not found in VISA INC.
tags not found in KKR & Co. Inc.
tags not found in Banco Santander (Brasil) S.A.
tags not found in MPLX LP
tags not found in Grayscale Bitcoin Trust ETF
tags not found in Anheuser-Busch InBev SA/NV
tags not found in Baker Hughes Co
tags not found in Cigna Group


In [ ]:
# Now i have 11 stocks left
# TODO: 

## More stuff

In [ ]:
# TODO: More tags for total number of shares
    # ifrs-full:WeightedAverageShares
    # ifrs-full:AdjustedWeightedAverageShares

    # us-gaap:WeightedAverageNumberDilutedSharesOutstandingAdjustment
    # us-gaap:WeightedAverageNumberOfSharesOutstandingBasic
    # us-gaap:WeightedAverageNumberOfDilutedSharesOutstanding
    # us-gaap:WeightedAverageNumberOfShareOutstandingBasicAndDiluted

    # us-gaap:PreferredStockSharesAuthorized
    # us-gaap:PreferredStockSharesIssued
    # us-gaap:PreferredStockSharesOutstanding

    # us-gaap:TreasuryStockShares
    # us-gaap:TreasuryStockCommonShares
    # us-gaap:TreasuryStockSharesAcquired